# Lab 6 · Popular LLMs — GPT, BERT, and T5 hands-on

> **Transformers teaching package — Lab.** Run top to bottom (Runtime → Run all). Read the notes, run the code,
> and finish the **Your turn 🧪** cell. Colors follow the *Neural Indigo* palette from the slides.


## What you'll build
Drive all three landmark architectures through the `transformers` **pipeline** API and see how their design
choices show up in what they're good at.

| Model | Architecture | Objective | Superpower |
|---|---|---|---|
| **BERT** | encoder-only | MLM | understanding / labeling |
| **GPT-2** | decoder-only | CLM | free-form generation |
| **T5** | encoder–decoder | span corruption | text-to-text transforms |

In [ ]:
!pip install -q transformers torch sentencepiece
from transformers import pipeline
import warnings; warnings.filterwarnings("ignore")
print("ready")

## 1 · BERT — fill in the blank (bidirectional understanding)

In [ ]:
fill = pipeline("fill-mask", model="bert-base-uncased")
for r in fill("artificial intelligence will [MASK] the world.")[:5]:
    print(round(r['score'],3), r['token_str'])

## 2 · GPT-2 — continue the text (causal generation)

In [ ]:
gen = pipeline("text-generation", model="gpt2")
out = gen("In the future, language models will",
          max_new_tokens=40, do_sample=True, temperature=0.8, top_p=0.9,
          num_return_sequences=1, pad_token_id=50256)
print(out[0]['generated_text'])

## 3 · T5 — text-to-text (summarize, translate, answer)

In [ ]:
t5 = pipeline("text2text-generation", model="t5-small")

article = ("The James Webb Space Telescope has captured its deepest image of the early universe yet, "
           "revealing thousands of galaxies formed shortly after the Big Bang. Astronomers say the data "
           "will help them understand how the first stars and galaxies came together.")
print("SUMMARY:", t5("summarize: " + article, max_new_tokens=40)[0]['generated_text'])
print("TRANSLATE:", t5("translate English to German: The cat drinks the milk.")[0]['generated_text'])

## 4 · Same task, three ways
Sentiment of a sentence — notice how each model's shape dictates *how* you ask.

In [ ]:
sent = "This movie was an absolute masterpiece."
# BERT-style: a fine-tuned classifier head
clf = pipeline("sentiment-analysis")   # DistilBERT fine-tuned on SST-2
print("BERT-style classifier:", clf(sent)[0])
# T5-style: frame it as text-to-text
print("T5 text-to-text:", t5(f"sst2 sentence: {sent}", max_new_tokens=5)[0]['generated_text'])
# GPT-style: prompt and read the continuation
print("GPT-style prompt:", gen(f"Review: {sent}\nSentiment:", max_new_tokens=3, pad_token_id=50256)[0]['generated_text'].splitlines()[-1])

## Your turn 🧪
1. Ask **T5** to do `question: ... context: ...` extractive QA.
2. Compare **GPT-2** generation at `temperature=0.3` vs `1.2` — what changes?
3. Use **BERT** `fill-mask` to probe world knowledge: `"The [MASK] is the largest planet."`
4. Time each pipeline. Which architecture is cheapest for classification vs generation, and why?

In [ ]:
# Your turn:
print(fill("the [MASK] is the largest planet in our solar system.")[0])
print(t5("question: Who painted the Mona Lisa? context: The Mona Lisa was painted by Leonardo da Vinci.")[0]['generated_text'])